# Queries
This notebook contains a set of DMS query examples that can be used as templates.

In [2]:
import json
import os
import time
from collections import defaultdict
from pathlib import Path

from dotenv import load_dotenv

from cognite.client import CogniteClient, global_config
from cognite.client.config import ClientConfig
from cognite.client.credentials import OAuthClientCredentials
from cognite.client.data_classes.data_modeling import (
    EdgeListWithCursor,
    InstanceSort,
    NodeId,
    NodeListWithCursor,
    ViewId,
)
from cognite.client.data_classes.data_modeling.query import (
    NodeResultSetExpression,
    Query,
    QueryResult,
    Select,
    SourceSelector,
)
from cognite.client.data_classes.filters import (
    And,
    ContainsAll,
    Equals,
    HasData,
    Nested,
    Prefix,
    SpaceFilter,
)
from cognite.client.exceptions import CogniteAPIError
from tenacity import (
    retry,
    retry_if_exception,
    stop_after_attempt,
    wait_exponential_jitter,
)

## Basic setup

In [3]:
# Locate the .env file by walking up from the current working directory until we find one.
# Notebooks set cwd to wherever Jupyter was launched, so a fixed `parents[N]` is fragile.
def _find_env_file(start: Path | None = None) -> Path | None:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        env_path = candidate / ".env"
        if env_path.is_file():
            return env_path
    return None


env_file = _find_env_file()
if env_file is None:
    raise FileNotFoundError(
        "Could not find a .env file by walking up from cwd. "
        "Place a .env at the repo root or set the CDF_* / IDP_* env vars in your shell."
    )
print(f"Loading credentials from {env_file}")
load_dotenv(env_file, override=False)

project = os.getenv("CDF_PROJECT")
cluster = os.getenv("CDF_CLUSTER")
client_id = os.getenv("CDF_CLIENT_ID") or os.getenv("IDP_CLIENT_ID")
client_secret = os.getenv("CDF_CLIENT_SECRET") or os.getenv("IDP_CLIENT_SECRET")
tenant_id = os.getenv("CDF_TENANT_ID") or os.getenv("IDP_TENANT_ID")
base_url = os.getenv("CDF_BASE_URL") or os.getenv("CDF_URL") or (f"https://{cluster}.cognitedata.com" if cluster else None)

missing = [
    name for name, val in {
        "CDF_PROJECT": project,
        "CDF_CLIENT_ID / IDP_CLIENT_ID": client_id,
        "CDF_CLIENT_SECRET / IDP_CLIENT_SECRET": client_secret,
        "CDF_TENANT_ID / IDP_TENANT_ID": tenant_id,
        "CDF_BASE_URL / CDF_URL / CDF_CLUSTER": base_url,
    }.items()
    if not val
]
if missing:
    raise RuntimeError(
        f"Missing required environment variables: {', '.join(missing)}.\n"
        f"Checked .env: {env_file}\n"
        "Make sure these are defined there or exported in your shell."
    )

credentials = OAuthClientCredentials(
    token_url=f"https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token",
    client_id=client_id,
    client_secret=client_secret,
    scopes=[f"{base_url}/.default"],
)

# Increase SDK-level retry budget so transient 429/5xx are handled before they reach user code.
# The SDK already retries 429 and 5xx with backoff; we just give it more headroom.
global_config.max_retries = 10
global_config.max_retry_backoff = 30
global_config.disable_pypi_version_check = True

client = CogniteClient(ClientConfig(
    client_name="cdf-query-examples",
    project=project,
    credentials=credentials,
    base_url=base_url,
))

# Merge (don't overwrite) headers so SDK defaults like x-cdp-sdk / User-Agent are preserved.
# cdf-version: alpha unlocks alpha features on /query (e.g. the "debug" envelope used below).
client.config.headers = {**(client.config.headers or {}), "cdf-version": "alpha"}
print("Client configured with alpha features enabled and SDK retries tuned (max_retries=10).")

Loading credentials from C:\Users\JanIngeBergseth\OneDrive - Cognite AS\Documents\GitHub\library\.env
Client configured with alpha features enabled and SDK retries tuned (max_retries=10).


# Retry helpers (408 / 429 / 5xx)

These wrappers handle the two failure modes you'll hit most often when running DMS queries:

- **408 Request Timeout** — usually caused by oversized payloads (`properties=["*"]`), high `limit`, or expensive joins. The right long-term fix is to narrow the query; the retry only buys time.
- **429 Too Many Requests** — token-bucket throttling. Exponential backoff with jitter clears it.

The decorator below retries on `{408, 425, 429, 500, 502, 503, 504}` with jittered exponential backoff. The Cognite SDK also retries internally (configured above via `global_config`); these wrappers add a second layer for the user-facing call.

In [4]:
RETRYABLE_CODES = {408, 425, 429, 500, 502, 503, 504}


def is_retryable_exception(e: BaseException) -> bool:
    """Retry on transient transport / throttling errors."""
    return isinstance(e, CogniteAPIError) and e.code in RETRYABLE_CODES


retry_cognite = retry(
    reraise=True,
    stop=stop_after_attempt(5),
    retry=retry_if_exception(is_retryable_exception),
    wait=wait_exponential_jitter(initial=1, max=30, jitter=2),
)


@retry_cognite
def run_query(client: CogniteClient, query: Query) -> QueryResult:
    """Run a /query against the data model with retry on 408/429/5xx."""
    return client.data_modeling.instances.query(query=query)


@retry_cognite
def run_sync(client: CogniteClient, query: Query) -> QueryResult:
    """Run a /sync (cursor-based) query with retry on 408/429/5xx."""
    return client.data_modeling.instances.sync(query=query)


def log_api_error(e: CogniteAPIError) -> None:
    """Print enough context to correlate with CDF backend logs."""
    print(f"CogniteAPIError code={e.code} x_request_id={getattr(e, 'x_request_id', None)}: {e.message}")

# Setup

Define the instance space, the model space, the model version, and all `ViewId`s used by the queries below. Centralising these constants makes it trivial to retarget the notebook at a different deployment.

In [37]:
# Target model in this project: dm_dom_oil_and_gas : dm_oil_and_gas_domain_model / v1
# Adjust INST_SP if your instance space differs from the model space.
INST_SP = "inst_location"    # instance space
MODEL_SP_CDM = "cdf_cdm"  # model space (where views/containers live)
MODEL_SP_IDM = "cdf_idm"  # model space (where views/containers live)
MODEL_VERSION = "v1"

# Project views (resolved from `client.data_modeling.views.list(space=MODEL_SP)`).
# `Tag` is this model's "asset" equivalent (functional tag). There is no Activity view -
# closest analogues are WorkOrder / WorkOrderOperation.
asset_vid = ViewId(space=MODEL_SP_CDM, external_id="CogniteAsset", version=MODEL_VERSION)
ts_vid = ViewId(space=MODEL_SP_CDM, external_id="CogniteTimeSeries", version=MODEL_VERSION)
file_vid = ViewId(space=MODEL_SP_CDM, external_id="CogniteFile", version=MODEL_VERSION)
eq_vid = ViewId(space=MODEL_SP_CDM, external_id="CogniteEquipment", version=MODEL_VERSION)
wo_vid = ViewId(space=MODEL_SP_IDM, external_id="CogniteMaintenanceOrder", version=MODEL_VERSION)
wo_op_vid = ViewId(space=MODEL_SP_IDM, external_id="CogniteOperation", version=MODEL_VERSION)
notification_vid = ViewId(space=MODEL_SP_IDM, external_id="CogniteNotification", version=MODEL_VERSION)


Optional sanity check: list all views in the model space.

In [41]:
# Inspect the ViewIds defined in the Setup cell. For each one, retrieve the full
# view definition from CDF and print its properties so you know what's available
# for filter / select / sort. Driven by the variables in Setup so renaming a view
# in one place is enough.
view_ids = {
    "asset_vid": asset_vid,
    "ts_vid": ts_vid,
    "file_vid": file_vid,
    "eq_vid": eq_vid,
    "wo_vid": wo_vid,
    "wo_op_vid": wo_op_vid,
    "notification_vid": notification_vid,
}

retrieved = client.data_modeling.views.retrieve(list(view_ids.values()))
retrieved_by_id = {(v.space, v.external_id, v.version): v for v in retrieved}

# Cache property metadata per view for downstream query-design / debugging.
# Schema: view_properties[var_name] = {prop_name: {"type": ..., "is_list": bool,
#         "is_direct_relation": bool, "is_reverse_direct_relation": bool,
#         "container": (space, external_id) | None}}
view_properties: dict[str, dict[str, dict]] = {}

for var_name, vid in view_ids.items():
    view = retrieved_by_id.get((vid.space, vid.external_id, vid.version))
    if view is None:
        print(f"{var_name} -> {vid.space}:{vid.external_id}/{vid.version}  (NOT FOUND)")
        view_properties[var_name] = {}
        continue

    props: dict[str, dict] = {}
    for prop_name, prop in view.properties.items():
        prop_type = getattr(prop, "type", None)
        cls_name = prop.__class__.__name__
        container = getattr(prop, "container", None)
        props[prop_name] = {
            "type": str(prop_type) if prop_type is not None else cls_name,
            "is_list": getattr(prop, "list", False),
            "is_direct_relation": cls_name == "MappedPropertyApply" and "direct" in str(prop_type).lower()
                or "direct" in cls_name.lower(),
            "is_reverse_direct_relation": "ReverseDirectRelation" in cls_name,
            "container": (container.space, container.external_id) if container is not None else None,
        }
    view_properties[var_name] = props
    print(f"{var_name} -> {vid.space}:{vid.external_id}/{vid.version}  ({len(props)} properties)")

# Quick-access helpers for query design:
#   view_properties["asset_vid"]["parent"]            -> dict of metadata
#   [p for p, m in view_properties["asset_vid"].items() if m["is_direct_relation"]]
#   [p for p, m in view_properties["asset_vid"].items() if m["is_reverse_direct_relation"]]
#   [p for p, m in view_properties["asset_vid"].items() if m["is_list"]]


asset_vid -> cdf_cdm:CogniteAsset/v1  (23 properties)
ts_vid -> cdf_cdm:CogniteTimeSeries/v1  (19 properties)
file_vid -> cdf_cdm:CogniteFile/v1  (18 properties)
eq_vid -> cdf_cdm:CogniteEquipment/v1  (18 properties)
wo_vid -> cdf_idm:CogniteMaintenanceOrder/v1  (24 properties)
wo_op_vid -> cdf_idm:CogniteOperation/v1  (27 properties)
notification_vid -> cdf_idm:CogniteNotification/v1  (21 properties)


# Assets
## General asset queries

### List assets and sort based on their externalId

Listing assets works almost the same as in the case of legacy assets. The main difference is the **sources** argument, that allows to choose the properties that will be fetched, by selecting a view (or a list of views).

You can sort/filter either by using a property specified within a View or node/edge registry.
Sorting by created/updated time is not allowed as of now, due to performance considerations (too much reindexing on every instance update).

In [44]:
# Listing instances through a view. The `sources` argument selects which view's
# properties get returned; sort/filter can reference any property on that view
# or the node/edge registry (e.g. ("node", "externalId")). Sorting on
# createdTime/lastUpdatedTime is not allowed.
#
# We omit `space=` so we discover where Tags actually live, then print the
# distribution of spaces so you can pin INST_SP correctly. The data
# (instance) space is usually different from the model space.
assets = client.data_modeling.instances.list(
    sources=asset_vid,
    limit=500,
    # Equals filter against a single text property on the view.
    filter=Equals(
        property=asset_vid.as_property_ref("sourceContext"),
        value="cfihos_test",
    ),
    sort=InstanceSort(property=["node", "externalId"], direction="descending"),
)
print(f"Returned {len(assets)} Tag instances.")

from collections import Counter

space_counts = Counter(a.space for a in assets)
print("Spaces seen in this batch:", dict(space_counts))
if assets:
    sample = assets[0]
    print(f"\nSample Tag: space={sample.space!r} externalId={sample.external_id!r}")
    print(sample)

# Once you know the right instance space, set INST_SP in the Setup cell and add
# `space=INST_SP` back here:
# filter=Equals(["node", "space"], value=INST_SP)
# ...or filter by a direct relation (note the {space, externalId} value shape):
# filter=Equals(
#     property=asset_vid.as_property_ref("parent"),
#     value={"space": INST_SP, "externalId": "SITE_OSL"},
# )

Returned 500 Tag instances.
Spaces seen in this batch: {'inst_location': 500}

Sample Tag: space='inst_location' externalId='f4af554c-57aa-4043-ad2b-263aa9e34485'
{
    "space": "inst_location",
    "external_id": "f4af554c-57aa-4043-ad2b-263aa9e34485",
    "version": 6,
    "last_updated_time": "2026-03-23 08:58:53.319+00:00",
    "created_time": "2026-03-18 13:14:48.062+00:00",
    "instance_type": "node",
    "properties": {
        "cdf_cdm": {
            "CogniteAsset/v1": {
                "name": "Seawater Lift Pump A",
                "tags": [
                    "rotating-equipment"
                ],
                "description": "Centrifugal seawater lift pump train A",
                "path": [
                    {
                        "space": "inst_location",
                        "externalId": "VAL-PH"
                    },
                    {
                        "space": "inst_location",
                        "externalId": "VAL-PH-23"
                 

# Search endpoint

Search is tokenized full-text lookup (not regex; see the docs for the exact tokenization rules).

This endpoint targets the Elasticsearch backend where *every text property is indexed*, so it is typically more performant than `/query` for keyword-style lookups.

Caveat: search can only match `text`-typed properties.

In [47]:
# Pick a real name from the data first - hardcoded demo values from another
# project (like "CNY-AC") will silently return zero hits.
sample = client.data_modeling.instances.list(sources=asset_vid, space=INST_SP, limit=5)
if not sample:
    raise RuntimeError("No Tag instances exist - ingest data or query a different view.")

sample_name = None
for node in sample:
    props = node.properties.get(asset_vid, {})
    if props.get("name"):
        sample_name = props["name"]
        break

# Search by a substring of a real name (drop the last 2 chars to make it a partial match).
search_term = (sample_name or "")[:-2] or sample_name
print(f"Searching Tag for {search_term!r} (derived from sample Tag {sample_name!r})")

results = client.data_modeling.instances.search(
    view=asset_vid,
    space=INST_SP,
    limit=20,
    sort=InstanceSort(property=asset_vid.as_property_ref("name"), direction="descending"),
    query=search_term,
)
print(f"Hits: {len(results)}")
for r in results[:5]:
    print(r)

Searching Tag for 'Electrical Distributi' (derived from sample Tag 'Electrical Distribution')
Hits: 1
{
    "space": "inst_location",
    "external_id": "VAL-PH-23-EL",
    "version": 35,
    "last_updated_time": "2026-05-08 12:42:05.798+00:00",
    "created_time": "2026-03-18 13:14:46.171+00:00",
    "instance_type": "node",
    "properties": {
        "cdf_cdm": {
            "CogniteAsset/v1": {
                "path": [
                    {
                        "space": "inst_location",
                        "externalId": "VAL-PH"
                    },
                    {
                        "space": "inst_location",
                        "externalId": "VAL-PH-23"
                    },
                    {
                        "space": "inst_location",
                        "externalId": "VAL-PH-23-EL"
                    }
                ],
                "root": {
                    "space": "inst_location",
                    "externalId": "VAL-PH"
    

## Iterative listing

Using instances API you can fetch the instances in batches, to avoid timeouts and reduce memory load

In [50]:
# Stream the full result set in batches. Set limit=None for unbounded iteration
# (this example caps at 20000 only to keep the demo fast - remove for production use).
# chunk_size controls page size; 500-1000 is typically a good compromise between
# round-trip overhead and the 408 risk that comes with huge per-page payloads.

# Pick a real sourceContext value from the data instead of hardcoding one.
from collections import Counter

probe = client.data_modeling.instances.list(sources=asset_vid, space=INST_SP, limit=200)
source_contexts = Counter(
    (node.properties.get(asset_vid, {}) or {}).get("sourceContext")
    for node in probe
)
print("sourceContext distribution (top 5):", source_contexts.most_common(5))

source_context_value = next(
    (val for val, _ in source_contexts.most_common() if val), None
)
if source_context_value is None:
    print("No sourceContext values found - iterating without a filter instead.")
    extra_filter = None
else:
    print(f"Iterating Tags with sourceContext == {source_context_value!r}")
    extra_filter = Equals(
        property=asset_vid.as_property_ref("sourceContext"),
        value=source_context_value,
    )

for i, assets in enumerate(client.data_modeling.instances(
    chunk_size=500,
    instance_type="node",
    sources=asset_vid,
    space=INST_SP,
    limit=20000,
    sort=InstanceSort(property=["node", "externalId"], direction="descending"),
    filter=extra_filter,
)):
    print(f"Fetching batch {i + 1}")
    print(len(assets))

sourceContext distribution (top 5): [('cfihos_test', 198), (None, 2)]
Iterating Tags with sourceContext == 'cfihos_test'
Fetching batch 1
500
Fetching batch 2
500
Fetching batch 3
201


### Fetching assets with the query endpoint

Use `/query` when you need only a specific subset of properties of the retrieved instances. This query is equivalent to the `list()` example above.

In [51]:
# Discover a real sourceContext value in INST_SP so the query actually returns something.
from collections import Counter

probe = client.data_modeling.instances.list(sources=asset_vid, space=INST_SP, limit=200)
source_contexts = Counter(
    (node.properties.get(asset_vid, {}) or {}).get("sourceContext")
    for node in probe
)
print("sourceContext distribution (top 5):", source_contexts.most_common(5))

sourceContext = next((val for val, _ in source_contexts.most_common() if val), None)
if sourceContext is None:
    raise RuntimeError(
        "No Tag in INST_SP has a populated sourceContext - pick another filter property."
    )
print(f"Querying Tags with sourceContext == {sourceContext!r}")

query = Query(
    with_={  # FROM all Nodes WHERE space = INST_SP AND has data in asset view AND sourceContext = <value>
        "assets": NodeResultSetExpression(
            filter=And(
                Equals(["node", "space"], value={"parameter": "space"}),
                HasData(views=[asset_vid]),
                Equals(property=asset_vid.as_property_ref("sourceContext"), value={"parameter": "sourceContext"}),
            ),
            limit=500,
            sort=[InstanceSort(property=("node", "externalId"), direction="descending")],
        ),
    },
    select={  # SELECT name, parent, tags FROM assets
        "assets": Select(
            [SourceSelector(asset_vid, ["name", "parent", "tags"])],
        ),
    },
    parameters={"space": INST_SP, "sourceContext": sourceContext},
)
try:
    res = run_query(client, query)
    assets = res["assets"]
    print(len(assets))
    print(assets)
except CogniteAPIError as e:
    log_api_error(e)


sourceContext distribution (top 5): [('cfihos_test', 198), (None, 2)]
Querying Tags with sourceContext == 'cfihos_test'
500
[
    {
        "space": "inst_location",
        "external_id": "f4af554c-57aa-4043-ad2b-263aa9e34485",
        "version": 6,
        "last_updated_time": "2026-03-23 08:58:53.319+00:00",
        "created_time": "2026-03-18 13:14:48.062+00:00",
        "instance_type": "node",
        "properties": {
            "cdf_cdm": {
                "CogniteAsset/v1": {
                    "parent": {
                        "space": "inst_location",
                        "externalId": "VAL-PH-23-WT"
                    },
                    "name": "Seawater Lift Pump A",
                    "tags": [
                        "rotating-equipment"
                    ]
                }
            }
        }
    },
    {
        "space": "inst_location",
        "external_id": "e43970f2-a955-44c7-beac-8f66eb2ae85d",
        "version": 5,
        "last_updated_time": "

# Get subtree of an asset

Getting a subset of assets based on a root is a common use case. Use the 'path' property to extract all assets with a given node in their paths.

### With ContainsAll

In [ ]:
# Pick a real subtree root: any Tag that is referenced as another Tag's parent
# is guaranteed to have at least one child, so ContainsAll/Prefix on path will
# return something. Cached as `sub_tree_root` for the Prefix cell below too.
try:
    sub_tree_root
except NameError:
    probe = client.data_modeling.instances.list(sources=asset_vid, space=INST_SP, limit=200)
    sub_tree_root = None
    for node in probe:
        parent = (node.properties.get(asset_vid, {}) or {}).get("parent")
        if parent and parent.get("space") and parent.get("externalId"):
            sub_tree_root = NodeId(parent["space"], parent["externalId"])
            break
    if sub_tree_root is None:
        raise RuntimeError(
            "No Tag in INST_SP has a populated `parent` - cannot derive a subtree root."
        )
    print(f"Using subtree root: {sub_tree_root}")

start_time = time.time()
sub_tree_nodes = client.data_modeling.instances.list(
    sources=asset_vid,
    filter=ContainsAll(property=asset_vid.as_property_ref("path"), values=[sub_tree_root]),
    limit=500,
)
contains_time = time.time() - start_time
print(f"ContainsAll filter call took: {contains_time:.3f} seconds")
print(len(sub_tree_nodes))
print(sub_tree_nodes)

### With Prefix (recommended)

`Prefix` on `path` uses the prefix index and is typically much faster than `ContainsAll` for subtree extraction. Prefer `Prefix` whenever you have the root's full `path`. Use `ContainsAll` only when you need to match a node at an arbitrary position in the path.

In [53]:
# Reuse sub_tree_root if the ContainsAll cell above has been run; otherwise discover one.
try:
    sub_tree_root
except NameError:
    probe = client.data_modeling.instances.list(sources=asset_vid, space=INST_SP, limit=200)
    sub_tree_root = None
    for node in probe:
        parent = (node.properties.get(asset_vid, {}) or {}).get("parent")
        if parent and parent.get("space") and parent.get("externalId"):
            sub_tree_root = NodeId(parent["space"], parent["externalId"])
            break
    if sub_tree_root is None:
        raise RuntimeError(
            "No Tag in INST_SP has a populated `parent` - cannot derive a subtree root."
        )
    print(f"Using subtree root: {sub_tree_root}")

# Retrieve the root asset first to get its path (excluded from the timing below).
sub_tree_root_retrieved = client.data_modeling.instances.retrieve_nodes(
    sub_tree_root,
    sources=asset_vid,
)

start_time = time.time()
sub_tree_nodes_prefix = client.data_modeling.instances.list(
    sources=asset_vid,
    filter=Prefix(
        property=asset_vid.as_property_ref("path"),
        value=sub_tree_root_retrieved.properties.data[asset_vid]["path"],
    ),
    limit=500,
)
prefix_time = time.time() - start_time
print(f"Prefix filter call took: {prefix_time:.3f} seconds")
print(len(sub_tree_nodes_prefix))
print(sub_tree_nodes_prefix)

Using subtree root: NodeId(space='inst_location', external_id='VAL-PH-23')
Prefix filter call took: 0.184 seconds
90
[
    {
        "space": "inst_location",
        "external_id": "VAL-PH-23-EL",
        "version": 35,
        "last_updated_time": "2026-05-08 12:42:05.798+00:00",
        "created_time": "2026-03-18 13:14:46.171+00:00",
        "instance_type": "node",
        "properties": {
            "cdf_cdm": {
                "CogniteAsset/v1": {
                    "path": [
                        {
                            "space": "inst_location",
                            "externalId": "VAL-PH"
                        },
                        {
                            "space": "inst_location",
                            "externalId": "VAL-PH-23"
                        },
                        {
                            "space": "inst_location",
                            "externalId": "VAL-PH-23-EL"
                        }
                    ],
      

## Get multiple representations of an asset

As you know, a single instance may have its properties in multiple views. When querying, listing or retrieval, it's possible to get multiple sources (views) along with their properties.

In [55]:
asset_external_id = "VAL-PH-23-WT"
space = INST_SP
client.data_modeling.instances.retrieve_nodes(
    NodeId(space, asset_external_id),
    # NOTE: `cdf_cdm` is not installed in this project. The original example added
    # cognite_asset_vid and describable_vid here to retrieve the same node viewed
    # through CDM. Add additional view IDs (e.g. asset_vid and any view that 'implements'
    # it, like FunctionalLocation) here if you want multi-view hydration.
    sources=[asset_vid],
)

,value
space,inst_location
external_id,VAL-PH-23-WT
version,35
last_updated_time,2026-05-08 12:42:07.619000
created_time,2026-03-18 13:14:46.171000
instance_type,node
path,"[{'space': 'inst_location', 'externalId': 'VAL..."
root,"{'space': 'inst_location', 'externalId': 'VAL-..."
parent,"{'space': 'inst_location', 'externalId': 'VAL-..."
pathLastUpdatedTime,2026-03-23T08:58:53.319752+00:00


### The same call using query SDK

In [56]:
asset_external_id = "VAL-PH-23-WT"
# Note: SpaceFilter does not accept {"parameter": ...} - use Equals(("node", "space"), ...)
# if you need to parameterize space. See "parent / children" example below.
query = Query(
    with_={  # FROM all Nodes WHERE space = INST_SP and externalId = CNY-AC
        "assets": NodeResultSetExpression(
            filter=And(
                Equals(["node", "externalId"], value=asset_external_id),
                SpaceFilter(space=INST_SP),
            ),
        ),
    },
    select={
        # seeing the same instance through multiple views - request only the properties you need
        "assets": Select(
            [
                SourceSelector(asset_vid, ["name", "description", "tags", "parent"]),
                # CDM views (cognite_asset_vid, describable_vid) are not installed
                # in this project. Add other project view IDs here if needed.
            ],
        ),
    },
)
try:
    res = run_query(client, query)
    print(res["assets"])
except CogniteAPIError as e:
    log_api_error(e)


[
    {
        "space": "inst_location",
        "external_id": "VAL-PH-23-WT",
        "version": 35,
        "last_updated_time": "2026-05-08 12:42:07.619+00:00",
        "created_time": "2026-03-18 13:14:46.171+00:00",
        "instance_type": "node",
        "properties": {
            "cdf_cdm": {
                "CogniteAsset/v1": {
                    "parent": {
                        "space": "inst_location",
                        "externalId": "VAL-PH-23"
                    },
                    "name": "Water Treatment System",
                    "tags": [
                        "functional-location"
                    ],
                    "description": "Seawater and chemical injection"
                }
            }
        }
    }
]


## Get parent and/or children of an asset

### Note that you can use this logic for any kind of **single** direct relations (and their reverse). For example, you can retrieve the type of the asset (see below)

This way, you can traverse a graph using direct relations

In [58]:
# asset_eid = "WLL-6080740225"
asset_eid = "23-XX-9105"
# Tip: parameterise space with Equals(("node","space"), {"parameter": "space"}) - SpaceFilter
# does not currently accept {"parameter": "..."}.
ASSET_PROPS = ["name", "description", "source", "parent", "tags"]
query = Query(
    with_={  # FROM all Nodes WHERE space = INST_SP and externalId = CNY-AC
        "asset": NodeResultSetExpression(
            filter=And(
                Equals(property=("node", "externalId"), value=asset_eid),
                Equals(property=("node", "space"), value={"parameter": "space"}),
            ),
        ),
        "parent": NodeResultSetExpression(
            from_="asset",
            through=asset_vid.as_property_ref("parent"),
            direction="outwards",
        ),
        "children": NodeResultSetExpression(
            from_="asset",
            through=asset_vid.as_property_ref("parent"),
            direction="inwards",
        ),
        "further_children": NodeResultSetExpression(
            from_="children",
            through=asset_vid.as_property_ref("parent"),
            direction="inwards",
        ),
        "type": NodeResultSetExpression(
            from_="asset",
            through=asset_vid.as_property_ref("type"),
            direction="outwards",
        ),
    },
    select={
        "asset": Select([SourceSelector(source=asset_vid, properties=ASSET_PROPS)]),
        "parent": Select([SourceSelector(source=asset_vid, properties=ASSET_PROPS)]),
        "children": Select([SourceSelector(source=asset_vid, properties=ASSET_PROPS)]),
        "further_children": Select([SourceSelector(source=asset_vid, properties=ASSET_PROPS)]),
        "type": Select([SourceSelector(source=asset_vid, properties=ASSET_PROPS)]),
    },
    parameters={"space": INST_SP},
)
try:
    res = run_query(client, query)
    print(f"asset: {res['asset']}")
    print(f"parent: {res['parent']}")
    print(f"children: {res['children']} (count={len(res['children'])})")
    print(f"further_children: {res['further_children']} (count={len(res['further_children'])})")
    print(f"type: {res['type']}")
except CogniteAPIError as e:
    log_api_error(e)



asset: [
    {
        "space": "inst_location",
        "external_id": "23-XX-9105",
        "version": 25,
        "last_updated_time": "2026-05-08 12:42:06.625+00:00",
        "created_time": "2026-03-23 09:24:29.979+00:00",
        "instance_type": "node",
        "properties": {
            "cdf_cdm": {
                "CogniteAsset/v1": {
                    "parent": {
                        "space": "inst_location",
                        "externalId": "23-1ST STAGE COMPRESSION-PH"
                    },
                    "name": "23-XX-9105",
                    "tags": [
                        "tag"
                    ],
                    "description": "VRD - 1ST STAGE SUCTION/DISCHARGE COOLER SKID"
                }
            }
        }
    }
]
parent: [
    {
        "space": "inst_location",
        "external_id": "23-1ST STAGE COMPRESSION-PH",
        "version": 24,
        "last_updated_time": "2026-05-08 12:42:09.497+00:00",
        "created_time": "2026-03-

## Using the Nested filter

Nested filter allows to use property of the directly related View to filter the instances. The filter can be applied only to single direct relations. 

In [ ]:
query = Query(
    with_={
        "asset": NodeResultSetExpression(
            # Tags whose PARENT TAG has 'functional-location' in its tags property.
            # Why parent? Nested can only filter through single-valued direct relations.
            # On Tag, `parent` is the only single DR to another Tag.
            filter=Nested(
                scope=asset_vid.as_property_ref("parent"),
                filter=ContainsAll(
                    property=asset_vid.as_property_ref("tags"),
                    values=["functional-location"],
                ),
            ),
            limit=500,
        ),
        "equipment": NodeResultSetExpression(
            # FROM all Equipment WHERE asset (a direct relation to a Tag) has sourceContext == "cfihos_test"
            filter=Nested(
                scope=eq_vid.as_property_ref("asset"),
                filter=Equals(
                    property=asset_vid.as_property_ref("sourceContext"),
                    value="cfihos_test",
                ),
            ),
            limit=500,
        ),
    },
    select={
        "asset": Select(
            [SourceSelector(source=asset_vid, properties=["name", "description", "tags", "parent"])]
        ),
        "equipment": Select(
            [SourceSelector(source=eq_vid, properties=["name", "description", "tags", "asset"])]
        ),
    },
)
try:
    res = run_query(client, query)
    print(f"asset (count={len(res['asset'])}):")
    for n in res['asset']:
        props = n.properties.get(asset_vid, {}) or {}
        print(f"ExId={n.external_id}  name={props.get('name')!r}")
    print(f"equipment (count={len(res['equipment'])}):")
    for n in res['equipment']:
        props = n.properties.get(eq_vid, {}) or {}
        print(f"ExId={n.external_id}  name={props.get('name')!r}")
except CogniteAPIError as e:
    log_api_error(e)


asset (count=98):
  VAL-PH-23-EL  name='Electrical Distribution'
  VAL-PH-23-GC  name='Gas Compression System'
  VAL-PH-23-UT  name='Utilities System'
  VAL-PH-23-WT  name='Water Treatment System'
  VAL-PH-48  name='Area 48 Subsea Systems'
  626221d1-e8f3-4761-97bd-239e013c7f75  name='Module Support Structure'
  6b0eb178-2c4f-4fca-901b-2c59bf812582  name='Instrument Equipment Room'
  a4b91d72-0d1b-4902-a8d4-98dfcbeabf70  name='1st Stage Safety Relief Valve'
  cd5759fc-0fa0-4ce3-9831-47529722f738  name='2nd Stage Suction ESDV'
  0192bf8a-c9a6-4f47-b491-208e6c11e351  name='1st Stage Suction Line 16in'
  1d338ea3-7841-43fc-afe6-702966cbbffc  name='Seawater Lift Pump B'
  4d839ff3-3a8f-4258-81d5-08409cdf2475  name='Portable Gas Detector'
  6767827c-f362-4b04-b303-fb5669ecab2b  name='Subsea Manifold'
  a86482db-1947-43e7-9781-67c6bc38ad09  name='MV Motor 1st Stage Compressor Drive'
  e3e89763-dd36-4823-9738-17d9a4fecd02  name='PA/GA System Process Area'
  056df394-acc1-4b71-9897-0053e2aec3b

## The same query as a json object

In some cases you may need to use a json object instead of SDK for querying

In [69]:
# JSON representation of the Nested filter query
json_query = {
    "with": {
        "asset": { # identifier of the item to retrieve
            "limit": 1000, # default limit is 100
            "nodes": {  # equivalent to FROM all Nodes in the project WHERE 'parent' of instances with properties
                        # in the Asset view has 'tags' property with value 'Permanently Abandoned'
                "filter": {
                    "nested": {
                        # Direct relation to instances with properties in Asset view through 'parent' property
                        "scope": [
                            asset_vid.space,
                            f"{asset_vid.external_id}/{asset_vid.version}",
                            "parent",
                        ],
                        # Filter by 'tags' property in Asset view
                        "filter": {
                            "containsAll": {
                                "property": [
                                    asset_vid.space,
                                    f"{asset_vid.external_id}/{asset_vid.version}",
                                    "aliases",
                                ],
                                # Value to filter by
                                "values": ["AC"],
                            },
                        },
                    },
                },
            },
        },
        "equipment": {
            # equivalent to FROM all Nodes in the project WHERE 'asset' of instances with properties
            # in the Equipment view has a tag 'Permanently Abandoned'
            "limit": 1000, # default limit is 100
            "nodes": {
                "filter": {
                    "nested": {
                        # Direct relation to instances with properties in Equipment view through 'asset' property
                        "scope": [
                            eq_vid.space,
                            f"{eq_vid.external_id}/{eq_vid.version}",
                            "asset",
                        ],
                        "filter": {
                            # Filter by 'tags' property in Equipment view
                            "containsAll": {
                                "property": [
                                    eq_vid.space,
                                    f"{eq_vid.external_id}/{eq_vid.version}",
                                    "tags",
                                ],
                                "values": ["Permanently Abandoned"],
                            },
                        },
                    },
                },
            },
        },
    },
    "select": {
        "asset": {
            "sources": [
                {
                    "source": {
                        "type": "view",
                        "space": asset_vid.space,
                        "externalId": asset_vid.external_id,
                        "version": asset_vid.version,
                    },
                    "properties": ["*"],  # All properties
                },
            ],
        },
        "equipment": {
            "sources": [
                {
                    "source": {
                        "type": "view",
                        "space": eq_vid.space,
                        "externalId": eq_vid.external_id,
                        "version": eq_vid.version,
                    },
                    "properties": ["*"],  # All properties
                },
            ],
        },
    },
    "debug": {},
}
@retry_cognite
def _post_query(payload: dict) -> dict:
    """POST the raw JSON query, raising CogniteAPIError on HTTP errors so retry can kick in."""
    response = client.post(
        url=f"/api/v1/projects/{client.config.project}/models/instances/query",
        json=payload,
    )
    # client.post returns a requests.Response - convert non-2xx into CogniteAPIError
    if response.status_code >= 400:
        try:
            body = response.json()
            message = body.get("error", {}).get("message", response.text)
        except ValueError:
            message = response.text
        raise CogniteAPIError(message=message, code=response.status_code)
    return response.json()


try:
    body = _post_query(json_query)
    assets = body["items"]["asset"]
    equipments = body["items"]["equipment"]
    print(f"asset (count={len(assets)}): {assets}")
    print(f"equipment (count={len(equipments)}): {equipments}")
    print("debug:", body.get("debug"))
except CogniteAPIError as e:
    log_api_error(e)


asset (count=0): []
equipment (count=0): []
debug: {'notices': []}


# Timeseries, activities, files
## Retrive timeseries related to an asset
Activities and files can be returned the same way.

The main problem here is that there is no way to extract assets and then use them to find the related timeseries. It is not possible because
- the properties holding node references pointing to assets are lists of direct relations
- reverse lists of direct relations cannot be queried

If your use case requires traversing multiple nodes both ways and lists of direct relations do not fulfill the requirements - that's when you need edges. Another way is to chain the queries outside of 'query' structure (query -> get result -> use in next query)

In [ ]:
# Replace external_id with a real Tag external_id from your project.
# Tip: run the list-assets cell above and copy any Tag's externalId here.
asset_id = NodeId(space=INST_SP, external_id="PLTF-EW1003A (Prince)-811")
print(asset_id.dump(include_instance_type=False))
query = Query(
    with_={
        "timeseries": NodeResultSetExpression(
            filter=ContainsAll(property=ts_vid.as_property_ref("assets"), values={"parameter": "asset"}),
            limit=500,
        ),
    },
    select={
        "timeseries": Select(
            [
                SourceSelector(
                    source=ts_vid,
                    properties=["name", "description", "source", "unit", "assets", "equipment", "activities"],
                ),
            ],
        ),
    },
    parameters={"asset": [asset_id.dump(include_instance_type=False)]},
)
try:
    res = run_query(client, query)
    print(res["timeseries"])
    print(len(res["timeseries"]))
except CogniteAPIError as e:
    log_api_error(e)


## Retrieve activities of a timeseries and equipment related to these activities

In [ ]:
# Replace external_id with a real TimeSeriesData externalId from your project.
timeseries_id = NodeId(space=INST_SP, external_id="CUMULATIVE_BOE_PER_DAY_TS_6081740998")
query = Query(
    with_={
        "activities": NodeResultSetExpression(
            filter=ContainsAll(
                property=wo_vid.as_property_ref("timeSeries"),
                values={"parameter": "timeseries"},
            ),
            limit=100,
        ),
        "equipment_activities": NodeResultSetExpression(
            from_="activities",
            through=wo_vid.as_property_ref("equipment"),  # must be a property reference
            limit=10,
        ),
    },
    select={
        "activities": Select(
            [
                SourceSelector(
                    source=wo_vid,
                    properties=["name", "description", "source", "assets", "equipment"],
                ),
            ],
        ),
        "equipment_activities": Select(
            [
                SourceSelector(
                    source=eq_vid,
                    properties=["name", "description", "source"],
                ),
            ],
        ),
    },
    parameters={"timeseries": [timeseries_id.dump(include_instance_type=False)]},
)
try:
    res = run_query(client, query)
    print(res["activities"])
    print(res["equipment_activities"])
    print("returned activities:", len(res["activities"]))
    print("returned equipment activities:", len(res["equipment_activities"]))
except CogniteAPIError as e:
    log_api_error(e)

## Retrieve equipment associated with an asset

You can retrieve equipment related to an asset through the 'asset' property in the Equipment.
This is useful when trying to get the equipment instances associated with assets of a certain type or class
or extensions of CogniteAsset with some properties.

Not that it only works with Equipment - all other Asset entity relationships (to files, timeseries, activities)
are Reverse **Lists** of direct relations, meaning they cannot be traversed inwards. 

In [ ]:
# Adjust these to values that actually exist in your project. Try the "list assets"
# cell first to see real `tags` and the `equipmentClass` values present on Equipment.
asset_tags = ["AC901"]
equipment_class = "Casing"

query = Query(
    with_={
        "assets": NodeResultSetExpression(
            # FROM all Nodes WHERE tags contains all of asset_tags
            filter=ContainsAll(property=asset_vid.as_property_ref("tags"), values={"parameter": "asset_tags"}),
            limit=500,
        ),
        "equipment": NodeResultSetExpression(
            from_="assets",
            through=eq_vid.as_property_ref("asset"),
            direction="inwards",
            # Equipment has an `equipmentClass` text property in this model,
            # so we filter directly instead of using a Nested filter through
            # the `equipmentType` direct relation (which would require the
            # cdf_cdm:CogniteEquipmentType view to be installed).
            filter=Equals(
                property=eq_vid.as_property_ref("equipmentClass"),
                value={"parameter": "equipmentClass"},
            ),
            limit=500,
        ),
    },
    select={
        "assets": Select(
            [SourceSelector(source=asset_vid, properties=["name", "description", "tags"])]
        ),
        "equipment": Select(
            [SourceSelector(source=eq_vid, properties=["name", "description", "asset", "equipmentClass", "class", "type"])]
        ),
    },
    parameters={"equipmentClass": equipment_class, "asset_tags": asset_tags},
)
try:
    res = run_query(client, query)
    print(f"assets (count={len(res['assets'])}): {res['assets']}")
    print(f"equipment (count={len(res['equipment'])}): {res['equipment']}")
except CogniteAPIError as e:
    log_api_error(e)


# Using the cursor

For completion, the methods below can be used to paginate with the instantiated query.

Examples of usage and considerations are TBD

In [ ]:
# Cursor-based pagination for data model queries (inspired by the Yggdrasil team).
# Uses run_sync (retry-wrapped /sync) so transient 408/429/5xx are handled transparently.

def get_data(
    client: CogniteClient,
    query: Query,
    max_iterations: int | None = 100,
) -> tuple[dict[str, list[NodeListWithCursor | EdgeListWithCursor]], dict[str, str]]:
    """Cursor based pagination for data model queries.

    The query object's cursors are updated in-place so the same query can be resumed.

    Args:
        client: The Cognite client to use for making the query.
        query: The query to fetch data from CDF data model.
        max_iterations: Maximum number of pages to fetch. Use None or -1 for no limit.

    Returns:
        A tuple of (collected_data, final_cursors). final_cursors is empty when the
        result set is fully drained.
    """
    if any(c for c in (query.cursors or {}).values()):
        print("Cursors already set in query, continuing retrieval.")

    collected_data: dict[str, list] = defaultdict(list)
    current_iteration = 0
    if max_iterations is None or max_iterations == -1:
        max_iterations = float("inf")

    res = None
    while current_iteration < max_iterations:
        res = run_sync(client, query)

        if res is None:
            if not collected_data:
                print("No data returned, exiting loop.")
                return {}, {}
            print("Query failed, but returning collected data so far.")
            return collected_data, {}

        # Empty page across all selections = fully drained (cursor still kept for resume).
        if all(not res.data[selection] for selection in res.data):
            print("No more data available, exiting loop.")
            return collected_data, {}

        for selection in res.data:
            collected_data[selection].extend(res.data[selection])

        query.cursors = res.cursors
        current_iteration += 1

    print(f"Collected data for {current_iteration} iterations.")
    return collected_data, (res.cursors if res is not None else {})

In [ ]:
# The retry helpers (retry_cognite, run_query, run_sync, log_api_error) are defined
# at the top of this notebook, right after client setup. Use them for any new query:
#
#     try:
#         res = run_query(client, query)        # /query with retry
#         # or
#         res = run_sync(client, query)         # /sync with retry (for cursor pagination)
#     except CogniteAPIError as e:
#         log_api_error(e)
#
# For cursor-paginated traversal of large result sets, use get_data(client, query)
# defined in the cell above - it calls run_sync internally.